# Part 1.A &mdash; Incremental Improvement of Baseline GPT2

 Starting from the Lab 04 minimal GPT2, improvements are applied **one at a time** so each PPL change can be attributed to a single cause:

 | Step | What changes |
 |---:|---|
 | 0 | Baseline &mdash; tune learning rate only |
 | 1 | Hyperparameter optimisation (`d_model`, `n_heads`, `num_layers`, `ff_dim`) |
 | 2 | Dropout at 4 required locations |
 | 3 | Weight tying between `token_embed` and `lm_head` |

 **Target:** PPL < 250 at every step, each modification improving on the previous.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader
from functools import partial
import math
import copy
import pandas as pd
from tqdm.notebook import tqdm
from transformers import AutoTokenizer

In [2]:
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

Using device: cuda:0


In [3]:
import os, urllib.request

os.makedirs('dataset/PennTreeBank', exist_ok=True)
base = 'https://raw.githubusercontent.com/massimo-rizzoli/NLU-2026-Labs/main/labs/dataset/PennTreeBank'
for split in ['ptb.train.txt', 'ptb.valid.txt', 'ptb.test.txt']:
    dst = f'dataset/PennTreeBank/{split}'
    if not os.path.exists(dst):
        urllib.request.urlretrieve(f'{base}/{split}', dst)
        print(f'Downloaded {split}')
    else:
        print(f'Already exists: {split}')

Downloaded ptb.train.txt
Downloaded ptb.valid.txt
Downloaded ptb.test.txt


In [4]:
def read_file(path, eos_token='<eos>'):
    output = []
    with open(path, 'r') as f:
        for line in f.readlines():
            output.append(line.strip() + ' ' + eos_token)
    return output

train_raw = read_file('dataset/PennTreeBank/ptb.train.txt')
dev_raw   = read_file('dataset/PennTreeBank/ptb.valid.txt')
test_raw  = read_file('dataset/PennTreeBank/ptb.test.txt')
print(f'Train: {len(train_raw)} | Dev: {len(dev_raw)} | Test: {len(test_raw)}')

Train: 42068 | Dev: 3370 | Test: 3761


In [5]:
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
tokenizer.pad_token = tokenizer.eos_token
vocab_len = len(tokenizer)
print(f'Vocabulary size: {vocab_len}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Vocabulary size: 50257


In [6]:
class PennTreeBank(data.Dataset):
    def __init__(self, corpus):
        self.sents = list(corpus)

    def __len__(self):
        return len(self.sents)

    def __getitem__(self, idx):
        return self.sents[idx]


def collate_fn(batch, tokenizer, device):
    tokenized = tokenizer(batch, padding=True, return_tensors='pt')
    input_ids = tokenized.input_ids[:, :-1].detach().clone().to(device)
    labels    = tokenized.input_ids[:, 1:].detach().clone().to(device)
    n_tokens  = torch.sum(input_ids != tokenizer.pad_token_id)
    return input_ids, labels, n_tokens


train_dataset = PennTreeBank(train_raw)
dev_dataset   = PennTreeBank(dev_raw)
test_dataset  = PennTreeBank(test_raw)

train_loader = DataLoader(
    train_dataset, batch_size=32,
    collate_fn=partial(collate_fn, tokenizer=tokenizer, device=DEVICE),
    shuffle=True)
dev_loader = DataLoader(
    dev_dataset, batch_size=64,
    collate_fn=partial(collate_fn, tokenizer=tokenizer, device=DEVICE))
test_loader = DataLoader(
    test_dataset, batch_size=64,
    collate_fn=partial(collate_fn, tokenizer=tokenizer, device=DEVICE))
print('Dataloaders ready.')

Dataloaders ready.


## Model Architecture

One flexible `GPT2` class covers all four experiments:

- `dropout` (default `0.0` = off) is wired into **4 required locations**:
  1. After the token + positional embedding sum (`GPT2.forward`)
  2. After the softmax attention weights (`MultiHeadAttention`)
  3. After the attention output projection (`MultiHeadAttention`)
  4. After the last linear of feed-forward (`FeedForward`)
- `weight_tying=True` makes `lm_head` share `token_embed`'s weight matrix

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.h_dim   = d_model // n_heads

        self.w_q      = nn.Linear(d_model, d_model)
        self.w_k      = nn.Linear(d_model, d_model)
        self.w_v      = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.attn_dropout  = nn.Dropout(dropout)  # location 2: after softmax
        self.resid_dropout = nn.Dropout(dropout)  # location 3: after out_proj

    def forward(self, x, mask):
        B, L, d_model = x.size()

        q = self.w_q(x).view(B, L, self.n_heads, self.h_dim).transpose(1, 2)
        k = self.w_k(x).view(B, L, self.n_heads, self.h_dim).transpose(1, 2)
        v = self.w_v(x).view(B, L, self.n_heads, self.h_dim).transpose(1, 2)

        sim  = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.h_dim))
        sim  = sim.masked_fill(mask == 0, float('-inf'))
        attn = self.attn_dropout(F.softmax(sim, dim=-1))

        y = (attn @ v).transpose(1, 2).contiguous().view(B, L, d_model)
        return self.resid_dropout(self.out_proj(y))

In [8]:
class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout=0.0):
        super().__init__()
        self.linear1    = nn.Linear(d_model, hidden_dim)
        self.act        = nn.GELU()
        self.linear2    = nn.Linear(hidden_dim, d_model)
        self.ff_dropout = nn.Dropout(dropout)  # location 4: after last linear

    def forward(self, x):
        x = self.linear1(x)
        x = self.act(x)
        x = self.linear2(x)
        return self.ff_dropout(x)

In [9]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.0):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = FeedForward(d_model, ff_dim, dropout)

    def forward(self, x, mask):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x

In [10]:
class GPT2(nn.Module):
    def __init__(
        self,
        vocab_size,
        pos_emb_size=1024,
        d_model=768,
        n_heads=12,
        num_layers=12,
        ff_dim=3072,
        dropout=0.0,
        weight_tying=False,
    ):
        super().__init__()
        self.pos_emb_size = pos_emb_size

        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed   = nn.Embedding(pos_emb_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)  # location 1: after embedding sum

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.ln_f    = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        if weight_tying:
            self.lm_head.weight = self.token_embed.weight

        mask = torch.tril(torch.ones(pos_emb_size, pos_emb_size)).unsqueeze(0).unsqueeze(0)
        self.register_buffer('mask', mask)

    def forward(self, idx):
        B, L = idx.shape
        assert L <= self.pos_emb_size
        pos = torch.arange(L, device=idx.device)
        x   = self.emb_dropout(self.token_embed(idx) + self.pos_embed(pos))
        mask = self.mask[:, :, :L, :L]
        for block in self.blocks:
            x = block(x, mask)
        return self.lm_head(self.ln_f(x))


print('Architecture defined.')

Architecture defined.


## Training Utilities

In [11]:
def train_loop(data, optimizer, criterion, model):
    model.train()
    loss_array, n_tokens_list = [], []
    pbar = tqdm(data, desc='Training', leave=False)
    for i, (input_ids, labels, n_tokens) in enumerate(pbar):
        optimizer.zero_grad()
        output = model(input_ids)
        loss   = criterion(output.permute(0, 2, 1), labels)
        loss_array.append(loss.item() * n_tokens)
        n_tokens_list.append(n_tokens)
        loss.backward()
        optimizer.step()
        if i % 100 == 0:
            pbar.set_postfix(loss=(sum(loss_array) / sum(n_tokens_list)).item())
    return sum(loss_array) / sum(n_tokens_list)


def eval_loop(data, eval_criterion, model):
    model.eval()
    loss_array, n_tokens_list = [], []
    with torch.no_grad():
        for input_ids, labels, n_tokens in tqdm(data, desc='Evaluating', leave=False):
            output = model(input_ids)
            loss   = eval_criterion(output.permute(0, 2, 1), labels)
            loss_array.append(loss.item() * n_tokens)
            n_tokens_list.append(n_tokens)
    avg_loss = sum(loss_array) / sum(n_tokens_list)
    return math.exp(avg_loss), avg_loss


def init_weights(mat):
    for m in mat.modules():
        if type(m) is nn.Linear:
            torch.nn.init.uniform_(m.weight, -0.01, 0.01)
            if m.bias is not None:
                m.bias.data.fill_(0.01)

In [12]:
def run_experiment(
    name,
    lr,
    d_model=20,
    n_heads=1,
    num_layers=1,
    ff_dim=20,
    dropout=0.0,
    weight_tying=False,
    n_epochs=50,
    patience=3,
):
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'Experiment : {name}')
    print(f'  lr={lr}  d_model={d_model}  n_heads={n_heads}  num_layers={num_layers}  ff_dim={ff_dim}')
    print(f'  dropout={dropout}  weight_tying={weight_tying}')
    print(sep)

    model = GPT2(
        vocab_len,
        pos_emb_size=1024,
        d_model=d_model,
        n_heads=n_heads,
        num_layers=num_layers,
        ff_dim=ff_dim,
        dropout=dropout,
        weight_tying=weight_tying,
    ).to(DEVICE)
    model.apply(init_weights)

    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {n_params:,}')

    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    best_ppl, best_model, pat = math.inf, None, patience

    pbar = tqdm(range(n_epochs), desc='Epochs')
    for epoch in pbar:
        train_loop(train_loader, optimizer, criterion, model)
        ppl_dev, _ = eval_loop(dev_loader, criterion, model)
        pbar.set_description(f'Dev PPL: {ppl_dev:.2f}')
        if ppl_dev < best_ppl:
            best_ppl   = ppl_dev
            best_model = copy.deepcopy(model).to('cpu')
            pat        = patience
        else:
            pat -= 1
        if pat <= 0:
            break

    best_model.to(DEVICE)
    ppl_test, _ = eval_loop(test_loader, criterion, best_model)
    best_model.to('cpu')

    print(f'  Best Dev PPL : {best_ppl:.2f}')
    print(f'  Test PPL     : {ppl_test:.2f}')

    results.append({
        'name'        : name,
        'lr'          : lr,
        'd_model'     : d_model,
        'n_heads'     : n_heads,
        'num_layers'  : num_layers,
        'ff_dim'      : ff_dim,
        'dropout'     : dropout,
        'weight_tying': weight_tying,
        'params'      : n_params,
        'dev_ppl'     : round(best_ppl, 2),
        'test_ppl'    : round(ppl_test, 2),
    })
    return best_model, best_ppl, ppl_test

In [13]:
# Re-run this cell only if you want to reset the results table
results = []

## Step 0  Baseline: Learning Rate Search

Architecture is fixed at the  minimal size:
```
d_model=20, n_heads=1, num_layers=1, ff_dim=20, dropout=0.0, weight_tying=False
```
#Only the learning rate changes. `lr=0.1` (the lab04 default) is too high for AdamW.
The best dev PPL here becomes the **baseline** to beat in Step 1.

In [ ]:
for lr in [1e-3, 5e-4, 1e-4]:
    run_experiment(
        name=f'Baseline lr={lr}',
        lr=lr,
        d_model=20, n_heads=1, num_layers=1, ff_dim=20,
        dropout=0.0, weight_tying=False,
        n_epochs=5
    )


Experiment : Baseline lr=0.001
  lr=0.001  d_model=20  n_heads=1  num_layers=1  ff_dim=20
  dropout=0.0  weight_tying=False
  Parameters: 2,033,400


Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Training:   0%|          | 0/1315 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/53 [00:00<?, ?it/s]

Training:   0%|          | 0/1315 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/53 [00:00<?, ?it/s]

Training:   0%|          | 0/1315 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/53 [00:00<?, ?it/s]

Training:   0%|          | 0/1315 [00:00<?, ?it/s]

## Step 1 Hyperparameter Optimisation

Using the best `lr` from Step 0, vary **one hyperparameter at a time**.
After each sub-step, note the best value and update the `best_*` variables in the next cell.

> Constraint: `d_model` must be divisible by `n_heads`. When scaling up significantly, the learning rate may also need adjustment.

In [ ]:
# --- Step 1a: d_model ---
# UPDATE best_lr from Step 0
best_lr = 1e-3  # UPDATE

for d_model in [64, 128, 256]:
    run_experiment(
        name=f'd_model={d_model}',
        lr=best_lr,
        d_model=d_model, n_heads=1, num_layers=1, ff_dim=20,
        dropout=0.0, weight_tying=False,
    )

In [ ]:
# --- Step 1b: n_heads ---
# n_heads must divide d_model evenly.
# UPDATE best values from Step 1a
best_lr      = 1e-3  # UPDATE
best_d_model = 128   # UPDATE

for n_heads in [2, 4]:
    if best_d_model % n_heads == 0:
        run_experiment(
            name=f'n_heads={n_heads}',
            lr=best_lr,
            d_model=best_d_model, n_heads=n_heads, num_layers=1, ff_dim=20,
            dropout=0.0, weight_tying=False,
        )

In [ ]:
# --- Step 1c: num_layers ---
# UPDATE best values from Steps 1a-1b
best_lr      = 1e-3  # UPDATE
best_d_model = 128   # UPDATE
best_n_heads = 2     # UPDATE

for num_layers in [2, 3, 4]:
    run_experiment(
        name=f'num_layers={num_layers}',
        lr=best_lr,
        d_model=best_d_model, n_heads=best_n_heads, num_layers=num_layers, ff_dim=20,
        dropout=0.0, weight_tying=False,
    )

In [ ]:
# --- Step 1d: ff_dim ---
# Standard ratios relative to d_model: x2 and x4
# UPDATE best values from Steps 1a-1c
best_lr         = 1e-3  # UPDATE
best_d_model    = 128   # UPDATE
best_n_heads    = 2     # UPDATE
best_num_layers = 2     # UPDATE

for ff_dim in [best_d_model * 2, best_d_model * 4]:
    run_experiment(
        name=f'ff_dim={ff_dim}',
        lr=best_lr,
        d_model=best_d_model, n_heads=best_n_heads, num_layers=best_num_layers, ff_dim=ff_dim,
        dropout=0.0, weight_tying=False,
    )

## Step 2 â€” Dropout

Dropout randomly zeroes activations during training, forcing the model to learn redundant representations (regularisation).

It is added at the **4 required locations** (already wired into the model class â€” set `dropout > 0` to activate them):
1. After token + positional embedding sum
2. After the softmax attention weights
3. After the attention output projection
4. After the last linear of feed-forward

Only `dropout` changes from Step 1; everything else stays at the best config found above.

In [ ]:
# --- Step 2: Dropout ---
# UPDATE all best_* values from Step 1
best_lr         = 1e-3   # UPDATE
best_d_model    = 128    # UPDATE
best_n_heads    = 2      # UPDATE
best_num_layers = 2      # UPDATE
best_ff_dim     = 256    # UPDATE

for dropout in [0.1, 0.2, 0.3]:
    run_experiment(
        name=f'dropout={dropout}',
        lr=best_lr,
        d_model=best_d_model, n_heads=best_n_heads, num_layers=best_num_layers, ff_dim=best_ff_dim,
        dropout=dropout, weight_tying=False,
    )

## Step 3 â€” Weight Tying

Weight tying shares the token embedding matrix (`token_embed.weight`) with the output projection (`lm_head.weight`). Both have shape `(vocab_size, d_model)`, so they are compatible.

Benefits:
- Reduces parameters by one `vocab_size Ã— d_model` matrix
- Encourages consistent input/output token representations (regularisation)

Only `weight_tying=True` changes; all other values match the best config from Step 2.

In [ ]:
# --- Step 3: Weight Tying ---
# UPDATE all best_* values from Step 2
best_lr         = 1e-3   # UPDATE
best_d_model    = 128    # UPDATE
best_n_heads    = 2      # UPDATE
best_num_layers = 2      # UPDATE
best_ff_dim     = 256    # UPDATE
best_dropout    = 0.1    # UPDATE

best_model, _, _ = run_experiment(
    name='Weight tying',
    lr=best_lr,
    d_model=best_d_model, n_heads=best_n_heads, num_layers=best_num_layers, ff_dim=best_ff_dim,
    dropout=best_dropout, weight_tying=True,
)

# Sanity check: both tensors must be the same object in memory
best_model.to(DEVICE)
assert best_model.lm_head.weight.data_ptr() == best_model.token_embed.weight.data_ptr()
print('Weight tying verified.')

## Results Summary

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
df = pd.DataFrame(results)
display(df)